# Tutorial 01 · Your Stroop effect and the price of precision

In the lecture you played a Stroop task and we asked nature three questions:
1. *Is there a Stroop effect?*
2. *How large is **your** Stroop effect?*
3. *How fast will you respond on the **next** trial?*

Here we answer them with the data of the whole class - and ask what each answer **costs** in trials. The analysis code is ready-made (`resources/stroop_analysis.py`); your job is the design questions. Throughout, we judge a design by the **variance of its estimates**.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

# add the repository root to the path (works from tutorials/ and tutorials/solutions/)
root = os.path.abspath("")
while not os.path.isdir(os.path.join(root, "resources")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

from resources.stroop_analysis import (ci_halfwidth, clean_trials, count_by_chance, participant_effects,
                                       probe_sequence_effects, random_sequence_effects, split_half_reliability,
                                       trials_until_significant, variance_components)
from resources.stroop_data import example_stroop_data, load_stroop_data

BLUE, RED = "#2f7bbf", "#ae0d36"
rng = np.random.default_rng(0)

## 1. The class data

All CSV files of the class go into `data/stroop/`. With fewer than 5 participants the notebook uses a **simulated example data set** in the same format instead.

Cleaning: we drop timeouts, errors and responses faster than 200 ms.

In [ ]:
MY_ID = "XXXX"   # put your participant ID here

data = load_stroop_data(block="1")
if data.empty or data["participant"].nunique() < 5:
    print("Fewer than 5 participants in data/stroop/ - using simulated example data.")
    data = example_stroop_data()
if MY_ID not in set(data["participant"]):
    MY_ID = data["participant"].iloc[0]
    print(f"ID not found - showing participant {MY_ID} instead.")

clean = clean_trials(data)
effects = participant_effects(clean)
print(f"{data['participant'].nunique()} participants, {len(clean)} of {len(data)} trials kept after cleaning")

## 2. Question 1: Is there a Stroop effect?

Everybody's Stroop effect with its 95% CI (yours in red), the class mean at the bottom - and how many trials it took until each effect was significant.

In [ ]:
e = effects.sort_values("effect").reset_index(drop=True)
class_mean, class_se = e["effect"].mean(), e["effect"].sem()
n_needed = clean.groupby("participant").apply(trials_until_significant, include_groups=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 0.22 * len(e) + 2))
colors = [RED if p == MY_ID else BLUE for p in e["participant"]]
ax1.hlines(e.index + 2, e["ci_low"], e["ci_high"], color=colors)
ax1.scatter(e["effect"], e.index + 2, color=colors, zorder=3)
ax1.errorbar(class_mean, 0, xerr=1.96 * class_se, fmt="D", color="black")
ax1.axvline(0, color="grey", lw=1)
ax1.set_yticks([0] + list(e.index + 2), ["class mean"] + list(e["participant"]))
ax1.set_xlabel("Stroop effect ± 95% CI (ms)")
ax2.hist(n_needed.dropna(), bins=range(0, 100, 6), color=BLUE)
ax2.set_xlabel("trials until p < 0.05")
ax2.set_ylabel("participants")
plt.tight_layout()
plt.show()

mine = n_needed[MY_ID]
print(f"class mean: {class_mean:.0f} ms ± {1.96 * class_se:.0f} ms (95% CI)")
print(f"you: {'not significant within your block' if np.isnan(mine) else f'significant after {mine:.0f} trials'}")
print(f"median: {n_needed.median():.0f} trials · never significant: {n_needed.isna().sum()} participants")

**Discuss:** The class mean is very precise, and most of you are significant after a few dozen trials. What did the remaining trials of the block add to the answer of Question 1?

*(Stopping at the first n with p < 0.05 is optional stopping - fine for describing data, never for deciding when to stop an experiment.)*

## 3. The price of precision

How precise would your Stroop effect be with $n$ trials? We resample your trials ($n$ with replacement, many times) and take the spread of the resulting effects as the uncertainty of an experiment with $n$ trials (the **bootstrap**, more in Session 2). Beyond the number of trials you actually played this is an extrapolation.

In [ ]:
mine = clean[clean["participant"] == MY_ID]
rt_mine, con_mine = mine["rt_ms"].to_numpy(), (mine["congruency"] == "congruent").to_numpy()

n_values = np.array([8, 16, 24, 32, 48, 64, 96, 128, 192, 256])
ci = np.array([ci_halfwidth(rt_mine, con_mine, n, rng=rng) for n in n_values])
k = np.sum(ci / np.sqrt(n_values)) / np.sum(1 / n_values)   # fit ci = k / sqrt(n)

plt.plot(n_values, ci, "o-", color=BLUE, label="resampled")
plt.plot(n_values, k / np.sqrt(n_values), "--", color="grey", label=r"$k / \sqrt{n}$")
plt.xlabel("number of trials")
plt.ylabel("95% CI half-width (ms)")
plt.legend()
plt.show()
print(f"k = {k:.0f} ms, i.e. ±{k / np.sqrt(len(mine)):.0f} ms with your {len(mine)} trials")

### Exercise 1
The CI half-width shrinks as $k / \sqrt{n}$. How many trials would **you** need for a CI of ±30 ms? And for ±10 ms? What does it cost to halve the CI?

In [ ]:
# ADD YOUR CODE HERE

**Discuss:** $k$ is large if your single trials are very variable (e.g. many slow responses). Besides running more trials, how could a design reduce the variance of a single trial?

## 4. Question 3: What happens next? The hidden probes

Your block was not purely random: **32 of the 96 trials** were hidden probes - `CCC` or `III`, followed by a congruent or incongruent target. Each of the four probes appeared twice, and inside a probe no word or ink colour was repeated.

The literature predicts a **congruency sequence effect** (Gratton et al., 1992): a smaller Stroop effect after incongruent trials. We measure it in two ways:
- **probes**: Stroop effect at the targets after `CCC` minus after `III`
- **random trials**: Stroop effect after a congruent minus after an incongruent previous trial - random sequences contain these pairs anyway

In [ ]:
probes = probe_sequence_effects(clean)
random_seq = random_sequence_effects(clean)
random_seq_clean = random_sequence_effects(clean, exclude_repetitions=True)

for label, values in [("probes (2 per probe type)", probes["sequence effect"]),
                      ("random trials", random_seq["sequence effect"]),
                      ("random trials, no feature repetitions", random_seq_clean["sequence effect"])]:
    print(f"{label:40s} {values.mean():5.0f} ms ± {values.sem():3.0f} ms (SE)")
for pattern in ["CCCI", "IIII"]:
    print(f"{pattern} by chance in the random trials: {count_by_chance(data, pattern).mean():.1f} per participant")

### Exercise 2
The SE of the probe estimate shrinks with $1/\sqrt{\text{repetitions}}$. How many repetitions of each probe would we need for an SE of 10 ms? How many trials would the block then have?

In [ ]:
# ADD YOUR CODE HERE

**Discuss:**
- Which design measures the sequence effect more precisely - the probes or the random trials? Why?
- What do the probes give us that the random trials do not? (Hint: feature repetitions, Mayr et al. 2003.)
- What does this suggest about random sampling as a design strategy? (Session 3.)

## 5. Question 2: How large is your Stroop effect? Reliability

To tell people apart, the measure has to rank them consistently. **Split-half reliability**: compute everybody's effect from the odd and from the even trials, correlate the two, and correct for the halved number of trials (Spearman-Brown, $2r/(1+r)$).

The lecture explained reliability by two sources of variance: real differences between people ($\sigma_{\text{between}}$) and the noise of single trials ($\sigma_{\text{trial}}$):

$$\text{reliability} = \frac{\sigma^2_{\text{between}}}{\sigma^2_{\text{between}} + 4\,\sigma^2_{\text{trial}} / k}$$

with $k$ trials per person. In the lecture we *assumed* $\sigma_{\text{between}} = 25$ ms and $\sigma_{\text{trial}} = 150$ ms - here we estimate them from our data.

In [ ]:
r_half, r_split = split_half_reliability(clean)
sd_between, sd_trial = variance_components(clean, effects)
k_trials = clean.groupby("participant").size().mean()
r_model = sd_between ** 2 / (sd_between ** 2 + 4 * sd_trial ** 2 / k_trials)

print(f"split-half reliability: {r_split:.2f}")
print(f"SD between people: {sd_between:.0f} ms · SD of a trial: {sd_trial:.0f} ms")
print(f"predicted reliability with {k_trials:.0f} trials: {r_model:.2f}")

### Exercise 3
Solve the formula for $k$: how many trials per person would we need for a reliability of 0.75 ("good")? How long would that block take (one trial ≈ 2 s)?

In [ ]:
# ADD YOUR CODE HERE

## 6. Take-away

- The **question** decides how many trials a design needs: Question 1 was answered after a few dozen trials, Questions 2 and 3 need far more.
- Precision grows only with $\sqrt{n}$ - so the only cheap way to more precision is **more informative trials**.
- A random design already contains many informative sequences - a strong baseline that any clever design has to beat.